##### Copyright 2025 Google LLC。

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# 使用 Hugging Face Transformers 和 QloRA 微調 Gemma

<table class="tfo-notebook-buttons" align="left"> <td>    <a target="_blank" href="https://ai.google.dev/gemma/docs/core/huggingface_text_finetune_qlora"><img src="https://ai.google.dev/static/site-assets/images/docs/notebook-site-button.png" height="32" width="32" />View on ai.google.dev</a>
</td> <td>    <a target="_blank" href="https://colab.research.google.com/github/google-gemma/cookbook/blob/main/docs/core/huggingface_text_finetune_qlora.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td> <td>    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/google-gemma/cookbook/blob/main/docs/core/huggingface_text_finetune_qlora.ipynb"><img src="https://www.kaggle.com/static/images/logos/kaggle-logo-transparent-300.png" height="32" width="70"/>Run in Kaggle</a>
</td> <td>    <a target="_blank" href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fgoogle-gemma%2Fcookbook%2Fmain%2Fdocs%2Fcore%2Fhuggingface_text_finetune_qlora.ipynb"><img src="https://ai.google.dev/images/cloud-icon.svg" width="40" />Open in Vertex AI</a>
</td> <td>    <a target="_blank" href="https://github.com/google-gemma/cookbook/blob/main/docs/core/huggingface_text_finetune_qlora.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
</td>
</table>

本指南將引導您了解如何使用 Hugging Face [Transformers](https://huggingface.co/docs/transformers/index) 和 [TRL](https://huggingface.co/docs/trl/index) 在自訂文字到 SQL dataset 上微調 Gemma。您將學到：
- 什麼是量化低階適應 (QLoRA)
- 設定開發環境
- 建立並準備 fine-tuning dataset
- 使用 TRL 和 SFTTrainer 微調Gemma
- 測試模型推論並產生 SQL 查詢

注意：本指南是為了在使用 16GB 和 Gemma 1B 的 NVIDIA T4 GPU 的 Google 合作帳戶上執行而創建的，但也可以進行調整以在更大的 GPU 和更大的模型上執行。
## 什麼是量化低階適應 (QLoRA)

本指南示範了[量化低階適應 (QLoRA)](https://arxiv.org/abs/2305.14314) 的使用，它是一種有效微調 LLM 的流行方法，因為它在保持高效能的同時減少了運算資源需求。在 QloRA 中，預訓練模型被量化為 4 位，並且權重被凍結。然後附加可訓練的適配器層（LoRA）並且僅訓練適配器層。之後，適配器權重可以與基本模型合併或保留為單獨的適配器。
## 設定開發環境

第一步是安裝 Hugging Face 庫，包括 TRL 和 datasets 來微調開放模型，包括不同的 RLHF 和對齊技術。

In [ ]:
# Install Pytorch & other libraries
%pip install torch tensorboard

# Install Transformers
%pip install "transformers>=5.10.1"

# Install Hugging Face libraries
%pip install datasets accelerate evaluate bitsandbytes trl peft protobuf sentencepiece

# COMMENT IN: if you are running on a GPU that supports BF16 data type and flash attn, such as NVIDIA L4 or NVIDIA A100
#%pip install flash-attn

_注意：如果您使用的是 Ampere 架構（例如 NVIDIA L4）或更新版本的 GPU，則可以使用 Flash Attention。 Flash Attention 是一種可顯著加快計算速度並將記憶體使用量從序列長度的二次方減少為線性方的方法，從而將訓練速度加快達 3 倍。了解更多信息，請訪問 [FlashAttention](https://github.com/Dao-AILab/flash-attention/tree/main)._
您需要有效的 Hugging Face token 才能發布您的模型。如果您在 Google Colab 內執行，則可以使用 Colab secrets 安全地使用 Hugging Face token，否則您可以直接在 `login` 方法中設定 token 。當您在訓練期間將模型推送到集線器時，請確保您的 token 也具有寫入權限。

In [ ]:
# Login into Hugging Face Hub
from huggingface_hub import login
login()

## 建立並準備 fine-tuning dataset

當fine-tuning 法學碩士時，了解您的用例和您想要解決的任務非常重要。這可以幫助您創建 dataset 來微調您的模型。如果您尚未定義用例，您可能需要返回繪圖板。
例如，本指南重點關注以下用例：
- 將自然語言微調為 SQL 模型，以便無縫整合到資料分析工具中。目標是大幅減少 SQL 查詢產生所需的時間和專業知識，甚至使非技術使用者也能從資料中提取有意義的見解。

文字到 SQL 對於 fine-tuning LLM 來說是一個很好的用例，因為這是一項複雜的任務，需要大量有關資料和 SQL 語言的（內部）知識。
一旦確定fine-tuning 是正確的解決方案，您就需要dataset 進行微調。 dataset 應該是您想要解決的任務的一組多樣化演示。有多種方法可以創建這樣的dataset，包括：
- 使用現有的開源dataset，例如[Spider](https://huggingface.co/datasets/spider)
- 使用法學碩士創建的合成 dataset，例如 [Alpaca](https://huggingface.co/datasets/tatsu-lab/alpaca)
- 使用人類創造的dataset，例如[Dolly](https://huggingface.co/datasets/databricks/databricks-dolly-15k)。
- 結合使用這些方法，例如 [Orca](https://huggingface.co/datasets/Open-Orca/OpenOrca)

每種方法都有其自身的優點和缺點，取決於預算、時間和品質要求。例如，使用現有的 dataset 是最簡單的，但可能無法針對您的特定用例進行定制，而使用領域專家可能是最準確的，但可能既耗時又昂貴。也可以組合多種方法來建立指令dataset，如[Orca: Progressive Learning from Complex Explanation Traces of GPT-4.](https://arxiv.org/abs/2306.02707)
本指南使用現有的 dataset ([philschmid/gretel-synthetic-text-to-sql](https://huggingface.co/datasets/philschmid/gretel-synthetic-text-to-sql))，這是一個高品質的合成文字到 SQL dataset，包括自然語言指令、模式定義、推論和相應的 SQL 查詢。
[Hugging Face TRL](https://huggingface.co/docs/trl/en/index) 支援會話 dataset 格式的自動模板化。這意味著您只需將 dataset 轉換為正確的 json 對象，`trl` 負責模板化並將其轉換為正確的格式。
```
{"messages": [{"role": "system", "content": "You are..."}, {"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}
{"messages": [{"role": "system", "content": "You are..."}, {"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}
{"messages": [{"role": "system", "content": "You are..."}, {"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}
```

[philschmid/gretel-synthetic-text-to-sql](https://huggingface.co/datasets/philschmid/gretel-synthetic-text-to-sql) 包含超過 100k 個樣本。為了保持指南較小，它被下採樣為僅使用 10,000 個樣本。
現在您可以使用 Hugging Face dataset library 載入 dataset 並建立 prompt 範本來組合自然語言指令、模式定義並為您的助理新增系統訊息。

In [ ]:
from datasets import load_dataset

# System message for the assistant
system_message = """You are a text to SQL query translator. Users will ask you questions in English and you will generate a SQL query based on the provided SCHEMA."""

# User prompt that combines the user query and the schema
user_prompt = """Given the <USER_QUERY> and the <SCHEMA>, generate the corresponding SQL command to retrieve the desired data, considering the query's syntax, semantics, and schema constraints.

<SCHEMA>
{context}
</SCHEMA>

<USER_QUERY>
{question}
</USER_QUERY>
"""
def create_conversation(sample):
  return {
    "messages": [
      {"role": "system", "content": system_message},
      {"role": "user", "content": user_prompt.format(question=sample["sql_prompt"], context=sample["sql_context"])},
      {"role": "assistant", "content": sample["sql"]}
    ]
  }

# Load dataset from the hub
dataset = load_dataset("philschmid/gretel-synthetic-text-to-sql", split="train")
dataset = dataset.shuffle().select(range(12500))

# Convert dataset to OAI messages
dataset = dataset.map(create_conversation, remove_columns=dataset.features,batched=False)
# split dataset into 80% training samples and 20% test samples
dataset = dataset.train_test_split(test_size=0.2)

# Print formatted user prompt
for item in dataset["train"][0]["messages"]:
  print(item)


README.md:   0%|          | 0.00/737 [00:00<?, ?B/s]

synthetic_text_to_sql_train.snappy.parqu(…):   0%|          | 0.00/32.4M [00:00<?, ?B/s]

synthetic_text_to_sql_test.snappy.parque(…):   0%|          | 0.00/1.90M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5851 [00:00<?, ? examples/s]

Map:   0%|          | 0/12500 [00:00<?, ? examples/s]

{'content': 'You are a text to SQL query translator. Users will ask you questions in English and you will generate a SQL query based on the provided SCHEMA.', 'role': 'system'}
{'content': "Given the <USER_QUERY> and the <SCHEMA>, generate the corresponding SQL command to retrieve the desired data, considering the query's syntax, semantics, and schema constraints.\n\n<SCHEMA>\nCREATE TABLE Menu (id INT PRIMARY KEY, name VARCHAR(255), category VARCHAR(255), price DECIMAL(5,2));\n</SCHEMA>\n\n<USER_QUERY>\nCalculate the average price of all menu items in the Vegan category\n</USER_QUERY>\n", 'role': 'user'}
{'content': "SELECT AVG(price) FROM Menu WHERE category = 'Vegan';", 'role': 'assistant'}


## 使用 TRL 和 SFTTrainer 微調Gemma

您現在已準備好微調您的模型。 Hugging Face TRL [SFTTrainer](https://huggingface.co/docs/trl/sft_trainer) 讓監督微調開放法學碩士變得簡單。 `SFTTrainer` 是`transformers` library 的`Trainer` 的子類，支援所有相同的功能，包括日誌記錄、評估和checkpointing，但增加了額外的生活品質功能，包括：
* dataset 格式化，包括會話格式和指令格式
* 僅針對完成情況進行培訓，忽略prompts
* 打包datasets 以實現更有效率的培訓
* 參數高效fine-tuning (PEFT) 支持，包括 QloRA
* 準備模型和tokenizer用於會話fine-tuning（例如添加特殊tokens）

以下程式碼從Hugging Face載入Gemma模型和tokenizer並初始化量化設定。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForImageTextToText, BitsAndBytesConfig

# Hugging Face model id
model_id = "google/gemma-4-E2B" # @param ["google/gemma-4-E2B","google/gemma-4-E4B","google/gemma-4-12B","google/gemma-4-31B","google/gemma-4-26B-A4B"] {"allow-input":true}

# Check if GPU benefits from bfloat16
if torch.cuda.get_device_capability()[0] >= 8:
    torch_dtype = torch.bfloat16
else:
    torch_dtype = torch.float16

# Define model init arguments
model_kwargs = dict(
    dtype=torch_dtype,
    device_map="auto", # Let torch decide how to load the model
)

# BitsAndBytesConfig: Enables 4-bit quantization to reduce model size/memory usage
model_kwargs["quantization_config"] = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=model_kwargs['dtype'],
    bnb_4bit_quant_storage=model_kwargs['dtype'],
)

# Load model and tokenizer
model = AutoModelForImageTextToText.from_pretrained(model_id, **model_kwargs)
tokenizer = AutoTokenizer.from_pretrained("google/gemma-4-E2B-it") # Load the Instruction Tokenizer to use the official Gemma template

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

`SFTTrainer` 支援與 `peft` 的內建集成，這使得使用 QLoRA 高效調整 LLM 變得簡單。您只需建立`LoraConfig`並提供給培訓師即可。

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=16,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
    modules_to_save=["lm_head", "embed_tokens"], # make sure to save the lm_head and embed_tokens as you train the special tokens
    ensure_weight_tying=True,
)

在開始訓練之前，您需要定義要在 `SFTConfig` 實例中使用的超參數。

In [ ]:
import torch
from trl import SFTConfig

args = SFTConfig(
    output_dir="gemma-text-to-sql",         # directory to save and repository id
    max_length=512,                         # max length for model and packing of the dataset
    num_train_epochs=3,                     # number of training epochs
    per_device_train_batch_size=1,          # batch size per device during training
    optim="adamw_torch_fused",              # use fused adamw optimizer
    logging_steps=10,                       # log every 10 steps
    save_strategy="epoch",                  # save checkpoint every epoch
    eval_strategy="epoch",                  # evaluate checkpoint every epoch
    learning_rate=5e-5,                     # learning rate
    fp16=True if model.dtype == torch.float16 else False,   # use float16 precision
    bf16=True if model.dtype == torch.bfloat16 else False,   # use bfloat16 precision
    max_grad_norm=0.3,                      # max gradient norm based on QLoRA paper
    lr_scheduler_type="constant",           # use constant learning rate scheduler
    push_to_hub=True,                           # push model to hub
    report_to="tensorboard",                # report metrics to tensorboard
    dataset_kwargs={
        "add_special_tokens": False, # Template with special tokens
        "append_concat_token": True, # Add EOS token as separator token between examples
    }
)

You now have every building block you need to create your `SFTTrainer` to start the training of your model.

In [ ]:
from trl import SFTTrainer

# Create Trainer object
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config,
    processing_class=tokenizer,
)

Tokenizing train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/2500 [00:00<?, ? examples/s]

透過呼叫 `train()` 方法開始訓練。

In [ ]:
# Start training, the model will be automatically saved to the Hub and the output directory
trainer.train()

# Save the final model again to the Hugging Face Hub
trainer.save_model()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Epoch,Training Loss,Validation Loss
1,0.536652,0.530056
2,0.430735,0.464053
3,0.386358,0.443147


在測試模型之前，請確保釋放記憶體。

In [ ]:
# free the memory again
del model
del trainer
torch.cuda.empty_cache()

使用QLoRA時，您僅訓練適配器而不是完整模型。這意味著在訓練期間保存模型時，您僅保存適配器權重，而不是完整模型。如果您想要儲存完整模型，以便更輕鬆地與 vLLM 或 TGI 等服務堆疊一起使用，您可以使用 `merge_and_unload` 方法將適配器權重合併到模型權重中，然後使用 `save_pretrained` 方法儲存模型。這會保存一個預設模型，可用於inference。
注意：當您要將適配器合併到模型中時，需要超過 30GB 的 CPU 記憶體。您可以跳過此步驟並繼續測試模型推論。

In [ ]:
from peft import PeftModel

# Load Model base model
model = AutoModelForImageTextToText.from_pretrained(model_id, low_cpu_mem_usage=True)

# Merge LoRA and base model and save
peft_model = PeftModel.from_pretrained(model, args.output_dir)
merged_model = peft_model.merge_and_unload()
merged_model.save_pretrained("merged_model", safe_serialization=True, max_shard_size="2GB")

processor = AutoTokenizer.from_pretrained(args.output_dir)
processor.save_pretrained("merged_model")

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/5 [00:00<?, ?it/s]

('merged_model/tokenizer_config.json',
 'merged_model/chat_template.jinja',
 'merged_model/tokenizer.json')

## 測試模型推論並產生 SQL 查詢

訓練完成後，您需要評估和測試您的模型。您可以從測試 dataset 載入不同的樣本，並在這些樣本上評估模型。
注意：評估生成式人工智慧模型並不是一項簡單的任務，因為一個輸入可以有多個正確的輸出。本指南僅關注手動評估和氛圍檢查。

In [ ]:
import torch
from transformers import pipeline

model_id = "merged_model"

# Load Model with PEFT adapter
model = AutoModelForImageTextToText.from_pretrained(
  model_id,
  device_map="auto",
  dtype="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

Loading weights:   0%|          | 0/2012 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.language_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


讓我們從測試 dataset 載入隨機樣本並產生 SQL 命令。

In [ ]:
from random import randint
import re
from transformers import pipeline, GenerationConfig

config = GenerationConfig.from_pretrained(model_id)
config.max_new_tokens = 256

# Load the model and tokenizer into the pipeline
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

# Load a random sample from the test dataset
rand_idx = randint(0, len(dataset["test"]))
test_sample = dataset["test"][rand_idx]

# Convert as test example into a prompt with the Gemma template
prompt = pipe.tokenizer.apply_chat_template(test_sample["messages"][:2], tokenize=False, add_generation_prompt=True)
print(prompt)

# Generate our SQL query.
outputs = pipe(prompt, generation_config=config)

# Extract the user query and original answer
print(f"Context:\n", re.search(r'<SCHEMA>\n(.*?)\n</SCHEMA>', test_sample['messages'][1]['content'], re.DOTALL).group(1).strip())
print(f"Query:\n", re.search(r'<USER_QUERY>\n(.*?)\n</USER_QUERY>', test_sample['messages'][1]['content'], re.DOTALL).group(1).strip())
print(f"Original Answer:\n{test_sample['messages'][2]['content']}")
print(f"Generated Answer:\n{outputs[0]['generated_text'][len(prompt):].strip()}")

<bos><|turn>system
You are a text to SQL query translator. Users will ask you questions in English and you will generate a SQL query based on the provided SCHEMA.<turn|>
<|turn>user
Given the <USER_QUERY> and the <SCHEMA>, generate the corresponding SQL command to retrieve the desired data, considering the query's syntax, semantics, and schema constraints.

<SCHEMA>
CREATE TABLE broadband_plans (plan_id INT, plan_name VARCHAR(255), download_speed INT, upload_speed INT, price DECIMAL(5,2));
</SCHEMA>

<USER_QUERY>
Delete a broadband plan from the 'broadband_plans' table
</USER_QUERY><turn|>
<|turn>model

Context:
 CREATE TABLE broadband_plans (plan_id INT, plan_name VARCHAR(255), download_speed INT, upload_speed INT, price DECIMAL(5,2));
Query:
 Delete a broadband plan from the 'broadband_plans' table
Original Answer:
DELETE FROM broadband_plans WHERE plan_id = 3001;
Generated Answer:
DELETE FROM broadband_plans
WHERE plan_name = 'Basic';


## 摘要與後續步驟

本教學介紹如何使用 TRL 和 QLoRA 微調 Gemma 模型。接下來查看以下文檔：
* 了解如何[使用 Gemma 模型產生文字](https://ai.google.dev/gemma/docs/get_started)。
* 了解如何[使用 Hugging Face Transformers 微調 Gemma 的視覺任務](https://ai.google.dev/gemma/docs/core/huggingface_vision_finetune_qlora)。
* 了解如何[在Gemma 模型上執行分佈式fine-tuning 和inference](https://ai.google.dev/gemma/docs/core/distributed_tuning)。
* 了解如何[使用 Gemma 和 Vertex AI 開放式模型](https://cloud.google.com/vertex-ai/docs/generative-ai/open-models/use-gemma)。
* 了解如何[使用KerasNLP 微調Gemma 並部署至Vertex AI](https://github.com/GoogleCloudPlatform/vertex-ai-samples/blob/main/notebooks/community/model_garden/model_garden_gemma_kerasnlp_to_vertexai.ipynb)。